# PHASE 3 - Deep Learning Approach

## Step 1 — Load MITRE STIX2 and Build Vocabularies

### 📝 Description

In this step, we load the **MITRE ATT&CK STIX2 dataset** and extract all the key objects — **techniques, tactics, groups, software, and platforms**.

Each object is then assigned a **numeric ID**, and these mappings are saved as JSON lookup files.  
For example:  
`tech2ix.json` maps technique IDs (like `T1059.001`) to numeric values.

#### Why we do this

- The MITRE dataset uses long text names and IDs (e.g., `T1059.001`), but  
  machine learning models only work with **numbers**, not strings.
- These lookup files act as a **vocabulary** that keeps everything consistent  
  when we build training data or run predictions later.

#### In short
We are building the **dictionary** our model will use to understand the MITRE world.


In [1]:
import json, os, re

ATTACK_PATH = "attack-stix-data/enterprise-attack/enterprise-attack.json"
ARTI = "artifacts"
os.makedirs(ARTI, exist_ok=True)

# Load the STIX bundle
with open(ATTACK_PATH, "r") as f:
    bundle = json.load(f)
objs = bundle.get("objects", [])

# Helper to get MITRE external IDs (like T1059)
def ext_id(o):
    for ref in o.get("external_references", []) or []:
        if ref.get("source_name") in ("mitre-attack", "mitre-ics-attack", "mitre-mobile-attack"):
            if "external_id" in ref:
                return ref["external_id"]
    return None

# Buckets for each MITRE object type
techniques = []   # attack-pattern
tactics    = []   # x-mitre-tactic
groups     = []   # intrusion-set
software   = []   # malware/tool
platforms  = set()

# Loop through and collect objects
for o in objs:
    t = o.get("type")
    if t == "attack-pattern":
        tid = ext_id(o)
        if tid:
            techniques.append({"id": tid, "name": o.get("name", ""), "is_sub": "." in tid})
            for p in o.get("x_mitre_platforms", []) or []:
                platforms.add(p)
    elif t == "x-mitre-tactic":
        taid = ext_id(o)
        if taid:
            tactics.append({"id": taid, "name": o.get("name", "")})
    elif t == "intrusion-set":
        gid = ext_id(o)
        if gid:
            groups.append({"id": gid, "name": o.get("name", "")})
    elif t in ("malware", "tool"):
        sid = ext_id(o)
        if sid:
            software.append({"id": sid, "name": o.get("name", "")})

# Make vocabularies (mapping from ID to index)
def make_vocab(items, key="id"):
    ids = sorted({it[key] for it in items})
    return {k: i for i, k in enumerate(ids)}, {i: k for i, k in enumerate(ids)}

tech2ix, ix2tech = make_vocab(techniques)
tac2ix, _ = make_vocab(tactics)
grp2ix, _ = make_vocab(groups)
sft2ix, _ = make_vocab(software)
plat2ix = {p: i for i, p in enumerate(sorted(platforms))}

# Save all vocabularies as JSON files
json.dump(tech2ix, open(f"{ARTI}/tech2ix.json", "w"))
json.dump(ix2tech, open(f"{ARTI}/ix2tech.json", "w"))
json.dump(tac2ix, open(f"{ARTI}/tac2ix.json", "w"))
json.dump(grp2ix, open(f"{ARTI}/grp2ix.json", "w"))
json.dump(sft2ix, open(f"{ARTI}/sft2ix.json", "w"))
json.dump(plat2ix, open(f"{ARTI}/plat2ix.json", "w"))

print("✅ Vocabularies saved!")
print(f"{len(tech2ix)} techniques, {len(tac2ix)} tactics, {len(grp2ix)} groups, {len(sft2ix)} software, {len(plat2ix)} platforms")


✅ Vocabularies saved!
823 techniques, 14 tactics, 181 groups, 758 software, 12 platforms


### 🧩 Step 2 — Build `kb_train.parquet` from MITRE

#### What we do

- Use the **vocab files** created in Step 1.  
- Read MITRE **relationships** such as:  
  - **Group uses → Technique**  
  - **Software uses → Technique**
- For each group or software, collect the **set of techniques** they use.
- For every technique in that set, create one training row:
  - `target_ix` → the technique to predict  
  - `ctx_ttps_ix` → the other techniques in the same set (context)  
  - plus: tactics, platforms, group/software IDs, popularity, and parent technique info.

#### Why we do it (short)

We need **training examples** that show which techniques tend to appear together in real attacks (as seen in MITRE).  
This becomes our **MITRE-only dataset**, used to train the **softmax model** before adding any org data.

#### ✅ Mini checklist

- [ ] Vocab JSON files exist in `artifacts/`  
- [ ] Extracted all “uses” relationships correctly  
- [ ] Technique sets are built per group/software  
- [ ] `artifacts/kb_train.parquet` is successfully created


In [2]:
import json, os, collections
import pandas as pd


DATA = "attack-stix-data/enterprise-attack/enterprise-attack.json"
ARTI = "artifacts"
os.makedirs(ARTI, exist_ok=True)

# ---- load bundle ----
with open(DATA, "r") as f:
    bundle = json.load(f)
objs = bundle.get("objects", [])

# ---- load vocabs from Step 1 ----
with open(f"{ARTI}/tech2ix.json","r") as f: tech2ix = json.load(f)
with open(f"{ARTI}/tac2ix.json","r")  as f: tac2ix  = json.load(f)
with open(f"{ARTI}/grp2ix.json","r")  as f: grp2ix  = json.load(f)
with open(f"{ARTI}/sft2ix.json","r")  as f: sft2ix  = json.load(f)
with open(f"{ARTI}/plat2ix.json","r") as f: plat2ix = json.load(f)

# ---- helpers to get MITRE external IDs ----
def ext_id(o):
    for ref in o.get("external_references", []) or []:
        if ref.get("source_name") in ("mitre-attack", "mitre-ics-attack", "mitre-mobile-attack"):
            if "external_id" in ref:
                return ref["external_id"]
    return None

# ---- index STIX objects by their internal stix id ----
by_id = {o.get("id"): o for o in objs if isinstance(o, dict) and o.get("id")}

# ---- technique metadata: platforms and tactics ----
tech_platforms = {}  # external TID -> set(platform names)
tech_tactics   = {}  # external TID -> set(tactic external IDs)
tactic_shortname_to_id = {}

for o in objs:
    if o.get("type") == "x-mitre-tactic":
        ta_ext = ext_id(o)
        short  = o.get("x_mitre_shortname")
        if ta_ext and short:
            tactic_shortname_to_id[short] = ta_ext

for o in objs:
    if o.get("type") == "attack-pattern":
        tid = ext_id(o)
        if not tid: 
            continue
        # platforms
        plats = set((o.get("x_mitre_platforms") or []))
        tech_platforms[tid] = plats
        # tactics via kill_chain_phases
        tacs = set()
        for kp in o.get("kill_chain_phases", []) or []:
            if kp.get("kill_chain_name") in ("mitre-attack","mitre-enterprise-attack","mitre-mobile-attack"):
                short = kp.get("phase_name")
                taid  = tactic_shortname_to_id.get(short)
                if taid:
                    tacs.add(taid)
        tech_tactics[tid] = tacs

# ---- build uses maps: groups/software -> set(tech external IDs) ----
group_to_techs   = collections.defaultdict(set)  # Gxxxx -> {Txxxx}
software_to_techs= collections.defaultdict(set)  # Sxxxx -> {Txxxx}

for o in objs:
    if o.get("type") != "relationship":
        continue
    rel = o.get("relationship_type")
    if rel != "uses":
        continue
    src = by_id.get(o.get("source_ref"))
    dst = by_id.get(o.get("target_ref"))
    if not src or not dst:
        continue

    # normalise: who uses what technique
    # intrusion-set/malware/tool uses attack-pattern
    src_type = src.get("type")
    dst_type = dst.get("type")

    if src_type in ("intrusion-set","malware","tool") and dst_type == "attack-pattern":
        owner_ext = ext_id(src)
        tech_ext  = ext_id(dst)
        if not owner_ext or not tech_ext:
            continue
        if src_type == "intrusion-set":
            group_to_techs[owner_ext].add(tech_ext)
        else:
            software_to_techs[owner_ext].add(tech_ext)

# ---- popularity: how often each technique appears across owners ----
pop_count = collections.Counter()
for g, Ts in group_to_techs.items():
    pop_count.update(Ts)
for s, Ts in software_to_techs.items():
    pop_count.update(Ts)

# ---- parent mapping for sub-techniques ----
def parent_of(tid):
    # e.g., T1059.001 -> T1059
    return tid.split(".")[0] if "." in tid else None

# ---- encode helpers ----
def enc_list(elems, vocab):
    out = []
    for e in elems:
        ix = vocab.get(e)
        if ix is not None:
            out.append(ix)
    return sorted(set(out))

def enc_single(e, vocab):
    return vocab.get(e, -1)

# ---- assemble training rows ----
rows = []

def emit_rows_from_owner(tech_set, group_ext=None, soft_ext=None):
    # compute shared context attrs from the set (excluding target later)
    for t in tech_set:
        ctx = set(tech_set) - {t}
        if not ctx:
            continue
        # tactics and platforms from the context techniques
        ctx_tacs = set()
        ctx_plats = set()
        for c in ctx:
            ctx_tacs |= tech_tactics.get(c, set())
            ctx_plats |= tech_platforms.get(c, set())

        row = {
            "target_ix": tech2ix.get(t, -999),
            "ctx_ttps_ix": enc_list(ctx, tech2ix),
            "group_ix": enc_single(group_ext, grp2ix) if group_ext else -1,
            "software_ix": enc_single(soft_ext, sft2ix) if soft_ext else -1,
            "tactic_ixs": enc_list(ctx_tacs, tac2ix),
            "platform_ixs": enc_list(ctx_plats, plat2ix),
            "popularity": float(pop_count.get(t, 0)),
            "parent_ix": tech2ix.get(parent_of(t), -1) if parent_of(t) else -1,
        }
        # only keep rows with valid target
        if row["target_ix"] != -999:
            rows.append(row)

# from groups
for g_ext, Tset in group_to_techs.items():
    emit_rows_from_owner(Tset, group_ext=g_ext, soft_ext=None)

# from software
for s_ext, Tset in software_to_techs.items():
    emit_rows_from_owner(Tset, group_ext=None, soft_ext=s_ext)

# ---- to DataFrame & save parquet ----
df = pd.DataFrame(rows)
# Lists are fine; you can explode later for batching. Requires pyarrow for parquet.
out_path = f"{ARTI}/kb_train.parquet"
df.to_parquet(out_path, index=False)
print(f"Saved {len(df)} rows to {out_path}")


Saved 14106 rows to artifacts/kb_train.parquet


In [3]:
df

,target_ix,ctx_ttps_ix,group_ix,software_ix,tactic_ixs,platform_ixs,popularity,parent_ix
0,252,"[6, 17, 23, 31, 36, 39, 80, 102, 142, 144, 148...",113,-1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]","[0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11]",160.0,-1
1,608,"[6, 17, 23, 31, 36, 39, 80, 102, 142, 144, 148...",113,-1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]","[0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11]",93.0,607
2,729,"[6, 17, 23, 31, 36, 39, 80, 102, 142, 144, 148...",113,-1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]","[0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11]",3.0,-1
3,713,"[6, 17, 23, 31, 36, 39, 80, 102, 142, 144, 148...",113,-1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]","[0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11]",19.0,712
4,289,"[6, 17, 23, 31, 36, 39, 80, 102, 142, 144, 148...",113,-1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]","[0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11]",4.0,-1
...,...,...,...,...,...,...,...,...
14101,201,"[241, 253, 404]",-1,485,"[4, 8, 10]","[1, 4, 5, 10, 11]",335.0,-1
14102,253,"[201, 241, 404]",-1,485,"[4, 6, 10]","[1, 4, 5, 10, 11]",163.0,-1
14103,241,"[201, 253, 404]",-1,485,"[4, 6, 8]","[1, 4, 5, 10, 11]",457.0,-1
14104,25,[626],-1,26,[4],[10],27.0,-1


In [4]:
import pandas as pd, json

df = pd.read_parquet("artifacts/kb_train.parquet")
with open("artifacts/tech2ix.json") as f:
    tech2ix = json.load(f)

covered = set(df["target_ix"].unique())
total = set(tech2ix.values())

print(f"{len(covered)} techniques covered out of {len(total)} total ({len(covered)/len(total)*100:.1f}%)")


564 techniques covered out of 823 total (68.5%)


### ✂️ Step 3 — Split into Train and Validation (Leave-Owner-Out)

#### What we do

- Split the dataset **by owners** — groups and software — not by rows.  
- Randomly pick about **20% of groups** and **20% of software** as validation owners.  
- Any row linked to those validation owners goes into the **validation set**.  
- All remaining rows go into the **training set**.

#### Why we do it (short)

This method checks whether the model can **generalise to unseen owners** — meaning new groups or software it hasn’t trained on.  
It’s a tougher and more realistic test than a simple random split of rows.


In [5]:
import os, json, random
import pandas as pd
import numpy as np

ARTI = "artifacts"
SEED = 42
VAL_FRAC_GROUP = 0.2
VAL_FRAC_SOFT  = 0.2

random.seed(SEED)
np.random.seed(SEED)

# load parquet from Step 2
kb_path = f"{ARTI}/kb_train.parquet"
df = pd.read_parquet(kb_path)

# load vocabs
with open(f"{ARTI}/grp2ix.json","r") as f: grp2ix = json.load(f)
with open(f"{ARTI}/sft2ix.json","r") as f: sft2ix = json.load(f)

# collect owner ids that appear in rows
groups_present = sorted(set(df["group_ix"].tolist()) - {-1})
soft_present   = sorted(set(df["software_ix"].tolist()) - {-1})

# pick validation owners (20% each, deterministic)
def sample_ids(ids, frac, seed):
    k = max(1, int(len(ids) * frac)) if len(ids) > 0 else 0
    rng = random.Random(seed)
    return set(rng.sample(ids, k)) if k > 0 else set()

val_groups = sample_ids(groups_present, VAL_FRAC_GROUP, SEED+1)
val_softs  = sample_ids(soft_present,  VAL_FRAC_SOFT,  SEED+2)

# assign split: row goes to VAL if its group_ix in val_groups OR software_ix in val_softs
is_val = (
    df["group_ix"].isin(val_groups) |
    df["software_ix"].isin(val_softs)
)

df_val   = df[is_val].reset_index(drop=True)
df_train = df[~is_val].reset_index(drop=True)

# sanity logs
print(f"Total rows: {len(df)}")
print(f"Train rows: {len(df_train)}")
print(f"Val rows:   {len(df_val)}")
print(f"Val groups (count): {len(val_groups)} of {len(groups_present)}")
print(f"Val software (count): {len(val_softs)} of {len(soft_present)}")

# save splits
train_path = f"{ARTI}/kb_train_split.parquet"
val_path   = f"{ARTI}/kb_val.parquet"
df_train.to_parquet(train_path, index=False)
df_val.to_parquet(val_path, index=False)

# also save the chosen owners for reproducibility
split_meta = {
    "seed": SEED,
    "val_groups": sorted(list(val_groups)),
    "val_software": sorted(list(val_softs)),
    "counts": {
        "rows_total": len(df),
        "rows_train": len(df_train),
        "rows_val": len(df_val),
        "groups_total": len(groups_present),
        "groups_val": len(val_groups),
        "software_total": len(soft_present),
        "software_val": len(val_softs),
    }
}
with open(f"{ARTI}/kb_split_meta.json","w") as f:
    json.dump(split_meta, f, indent=2)

print("Saved:")
print(" -", train_path)
print(" -", val_path)
print(" - artifacts/kb_split_meta.json")


Total rows: 14106
Train rows: 11248
Val rows:   2858
Val groups (count): 31 of 158
Val software (count): 143 of 718
Saved:
 - artifacts/kb_train_split.parquet
 - artifacts/kb_val.parquet
 - artifacts/kb_split_meta.json


### 🧩 Step 4 — Build the Softmax Model (and Get It Ready to Train)

#### What we do

- Build a **TensorFlow / Keras neural network** that predicts one missing technique.  
- The model takes multiple **context inputs**:
  - Group or Software ID  
  - Tactics (multi-hot vector)  
  - Platforms (multi-hot vector)  
  - Context TTPs (multi-hot vector showing which techniques co-occur)  
- Combine these inputs into one dense layer and output a **softmax** over all techniques.  
- Prepare **training data** using scikit-learn and pandas:
  - Convert list columns (like tactics or context TTPs) into multi-hot vectors.  
  - Build TensorFlow Datasets for training and validation.  
- Set up the **loss** as Sparse Categorical Cross-Entropy (with label smoothing handled by Keras automatically).  
- Add metrics like **Top-1**, **Top-5**, and **Top-10 Accuracy** to track performance.  
- Use callbacks for **early stopping** and **model checkpointing** while training.

#### Why we do it (short)

The model learns which techniques **usually appear together** in MITRE data.  
Once trained, we can give it an organisation’s known TTPs and it will **predict other techniques** that are likely missing or could appear in future attacks.  

This helps identify **defensive gaps** and **next likely threats**.


In [6]:
import os, json
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils import Bunch

# ---- paths ----
ARTI = "artifacts"
MODEL_OUT = "models"
os.makedirs(MODEL_OUT, exist_ok=True)

# ---- load vocab sizes ----
with open(f"{ARTI}/tech2ix.json","r") as f: tech2ix = json.load(f)
with open(f"{ARTI}/tac2ix.json","r")  as f: tac2ix  = json.load(f)
with open(f"{ARTI}/plat2ix.json","r") as f: plat2ix = json.load(f)
with open(f"{ARTI}/grp2ix.json","r")  as f: grp2ix  = json.load(f)
with open(f"{ARTI}/sft2ix.json","r")  as f: sft2ix  = json.load(f)

D_TEC = len(tech2ix)
D_TAC = len(tac2ix)
D_PLT = len(plat2ix)
D_G   = len(grp2ix)
D_S   = len(sft2ix)

# ---- small helpers ----
def ensure_list(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return list(x)
    if pd.isna(x):
        return []
    try:
        return list(x)
    except Exception:
        return [int(x)]

def multi_hot(indices, size):
    v = np.zeros(size, dtype=np.float32)
    for ix in indices or []:
        if isinstance(ix, (list, tuple)):  # safety if nested
            for j in ix:
                if 0 <= int(j) < size:
                    v[int(j)] = 1.0
        else:
            j = int(ix)
            if 0 <= j < size:
                v[j] = 1.0
    return v

def load_split(train_path=f"{ARTI}/kb_train_split.parquet", val_path=f"{ARTI}/kb_val.parquet"):
    train_df = pd.read_parquet(train_path)
    val_df   = pd.read_parquet(val_path)
    # normalise list columns
    for col in ["ctx_ttps_ix","tactic_ixs","platform_ixs"]:
        train_df[col] = train_df[col].apply(ensure_list)
        val_df[col]   = val_df[col].apply(ensure_list)
    return train_df, val_df

def df_to_arrays(df):
    g   = df["group_ix"].fillna(-1).astype("int32").to_numpy()
    s   = df["software_ix"].fillna(-1).astype("int32").to_numpy()
    # map -1 → 0 bucket; keep max to vocab size
    g = np.clip(g, 0, len(grp2ix)).astype("int32")
    s = np.clip(s, 0, len(sft2ix)).astype("int32")

    ctx = np.stack([multi_hot(xs, D_TEC) for xs in df["ctx_ttps_ix"]]).astype("float32")
    tac = np.stack([multi_hot(xs, D_TAC) for xs in df["tactic_ixs"]]).astype("float32")
    pl  = np.stack([multi_hot(xs, D_PLT) for xs in df["platform_ixs"]]).astype("float32")
    y   = df["target_ix"].astype("int32").to_numpy()
    return Bunch(group=g, software=s, ctx=ctx, tac=tac, plat=pl, y=y)


def make_tfds(arrs, batch=512, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((
        {
            "group_ix":    arrs.group,
            "software_ix": arrs.software,
            "ctx":         arrs.ctx,
            "tac":         arrs.tac,
            "plat":        arrs.plat
        },
        arrs.y
    ))
    if shuffle:
        ds = ds.shuffle(min(10000, arrs.y.shape[0]), seed=42, reshuffle_each_iteration=True)
    ds = ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds

# ---- model in Keras ----
def build_model(
    d_g=32, d_s=32, d_tac=16, d_plt=8, d_ctx=128, hidden=256,
    l2=1e-5, dropout=0.1
):
    inp_g   = tf.keras.Input(shape=(), dtype=tf.int32,   name="group_ix")
    inp_s   = tf.keras.Input(shape=(), dtype=tf.int32,   name="software_ix")
    inp_ctx = tf.keras.Input(shape=(D_TEC,), dtype=tf.float32, name="ctx")
    inp_tac = tf.keras.Input(shape=(D_TAC,), dtype=tf.float32, name="tac")
    inp_pl  = tf.keras.Input(shape=(D_PLT,), dtype=tf.float32, name="plat")

    emb_g  = tf.keras.layers.Embedding(
        input_dim=D_G+1, output_dim=d_g,
        embeddings_regularizer=tf.keras.regularizers.l2(l2),
        name="emb_group"
    )(inp_g)

    emb_s  = tf.keras.layers.Embedding(
        input_dim=D_S+1, output_dim=d_s,
        embeddings_regularizer=tf.keras.regularizers.l2(l2),
        name="emb_soft"
    )(inp_s)

    proj_ctx = tf.keras.layers.Dense(d_ctx, use_bias=False,
                                     kernel_regularizer=tf.keras.regularizers.l2(l2),
                                     name="proj_ctx")(inp_ctx)
    proj_tac = tf.keras.layers.Dense(d_tac, use_bias=False,
                                     kernel_regularizer=tf.keras.regularizers.l2(l2),
                                     name="proj_tac")(inp_tac)
    proj_pl  = tf.keras.layers.Dense(d_plt, use_bias=False,
                                     kernel_regularizer=tf.keras.regularizers.l2(l2),
                                     name="proj_pl")(inp_pl)

    concat = tf.keras.layers.Concatenate()([emb_g, emb_s, proj_ctx, proj_tac, proj_pl])
    h = tf.keras.layers.Dense(hidden, activation="relu",
                              kernel_regularizer=tf.keras.regularizers.l2(l2))(concat)
    h = tf.keras.layers.Dropout(dropout)(h)
    h = tf.keras.layers.Dense(128, activation="relu",
                              kernel_regularizer=tf.keras.regularizers.l2(l2))(h)

    logits = tf.keras.layers.Dense(D_TEC, name="logits")(h)

    model = tf.keras.Model(
        inputs={"group_ix": inp_g, "software_ix": inp_s, "ctx": inp_ctx, "tac": inp_tac, "plat": inp_pl},
        outputs=logits
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=1,  name="top1"),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5,  name="top5"),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=10, name="top10"),
        ]
    )
    return model


# ---- load data, build datasets ----
train_df, val_df = load_split()
train_arr = df_to_arrays(train_df)
val_arr   = df_to_arrays(val_df)

train_ds = make_tfds(train_arr, batch=512, shuffle=True)
val_ds   = make_tfds(val_arr,   batch=512, shuffle=False)

# ---- build and train (skeleton) ----
model = build_model()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_top10", mode="max", patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(MODEL_OUT, "kb_softmax_tf.keras"),
        monitor="val_top10", mode="max", save_best_only=True
    )
]

print(model.summary())

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

# ---- save final model (SavedModel / .keras) ----
model.save(os.path.join(MODEL_OUT, "kb_softmax_tf.keras"))
print("Saved model to:", os.path.join(MODEL_OUT, "kb_softmax_tf.keras"))


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ group_ix            │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ software_ix         │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ctx (InputLayer)    │ (None, 823)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tac (InputLayer)    │ (None, 14)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ plat (InputLayer)   │ (None, 12)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_group           │ (None, 32)        │      5,824 │ group_ix[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_soft            │ (None, 32)        │     24,288 │ software_ix[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ proj_ctx (Dense)    │ (None, 128)       │    105,344 │ ctx[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ proj_tac (Dense)    │ (None, 16)        │        224 │ tac[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ proj_pl (Dense)     │ (None, 8)         │         96 │ plat[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 216)       │          0 │ emb_group[0][0],  │
│ (Concatenate)       │                   │            │ emb_soft[0][0],   │
│                     │                   │            │ proj_ctx[0][0],   │
│                     │                   │            │ proj_tac[0][0],   │
│                     │                   │            │ proj_pl[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │     55,552 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ logits (Dense)      │ (None, 823)       │    106,167 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 330,391 (1.26 MB)

 Trainable params: 330,391 (1.26 MB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 6.1225 - top1: 0.0244 - top10: 0.1586 - top5: 0.0892 - val_loss: 5.5762 - val_top1: 0.0360 - val_top10: 0.2204 - val_top5: 0.1351
Epoch 2/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 5.2826 - top1: 0.0309 - top10: 0.2245 - top5: 0.1357 - val_loss: 5.2812 - val_top1: 0.0322 - val_top10: 0.2316 - val_top5: 0.1372
Epoch 3/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 5.1396 - top1: 0.0428 - top10: 0.2429 - top5: 0.1528 - val_loss: 5.2123 - val_top1: 0.0486 - val_top10: 0.2292 - val_top5: 0.1501
Epoch 4/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 5.0475 - top1: 0.0551 - top10: 0.2536 - top5: 0.1632 - val_loss: 5.1567 - val_top1: 0.0500 - val_top10: 0.2435 - val_top5: 0.1568
Epoch 5/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.9562 - top1: 0.0654 - top10: 0.2749 - top5: 0.1840 - val_loss: 5.1020 - val_top1: 0.0647 - val_top10: 0.2624 - val_top5: 0.1795
Epoch 6/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - lo

### 🚀 Step 5 — Train the Model and Save It

#### What we do

- Train the **TensorFlow/Keras softmax model** using the MITRE-only train and validation splits.  
- Apply **early stopping** to prevent overfitting and **model checkpointing** to save the best version.  
- Track **Top-K accuracy metrics** — specifically **Top-1**, **Top-5**, and **Top-10** — on the validation set.  
- Save both:
  - The **trained model** (for later use)  
  - A **training log CSV** with loss and accuracy trends.

#### Why we do it (short)

This step turns the MITRE knowledge base dataset into a **working predictor**.  
The saved model will later help with **organisation gap analysis** — predicting which TTPs are likely missing for a given org.


In [7]:
import os, json, pandas as pd, numpy as np
import tensorflow as tf

ARTI = "artifacts"
MODEL_OUT = "models"
LOGS_OUT = "artifacts"
os.makedirs(MODEL_OUT, exist_ok=True)
os.makedirs(LOGS_OUT, exist_ok=True)

# --- bring in helper pieces from Step 4 (reuse exactly) ---
# - load_split()
# - df_to_arrays()
# - make_tfds()
# - build_model()

# If you put them in a module, import instead:
# from kb_softmax_tf import load_split, df_to_arrays, make_tfds, build_model

# Load data
train_df, val_df = load_split()
train_arr = df_to_arrays(train_df)
val_arr   = df_to_arrays(val_df)

train_ds = make_tfds(train_arr, batch=512, shuffle=True)
val_ds   = make_tfds(val_arr,   batch=512, shuffle=False)

# Build model (with label smoothing inside the loss)
def build_model_with_ls(ls=0.05):
    m = build_model()
    m.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=1,  name="top1"),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5,  name="top5"),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=10, name="top10"),
        ]
    )
    return m

model = build_model_with_ls(ls=0.05)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_top10", mode="max", patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(MODEL_OUT, "kb_softmax_tf.keras"),
        monitor="val_top10", mode="max", save_best_only=True
    ),
    tf.keras.callbacks.CSVLogger(os.path.join(LOGS_OUT, "kb_train_log.csv"))
]

print(model.summary())

# Train
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

# Evaluate best model on validation
val_metrics = model.evaluate(val_ds, return_dict=True, verbose=0)
print("Validation metrics:", val_metrics)

# Save final model (even if same as checkpoint)
final_path = os.path.join(MODEL_OUT, "kb_softmax_tf_final.keras")
model.save(final_path)
print("Saved final model to:", final_path)

# Small sanity check: show a few Top-5 predictions vs. true labels
def preview_samples(ds, n_batches=1):
    for i, (xb, yb) in enumerate(ds.take(n_batches)):
        logits = model(xb, training=False)
        top5 = tf.math.top_k(logits, k=5).indices.numpy()
        y_np = yb.numpy()
        for j in range(min(5, y_np.shape[0])):
            print(f"Sample {j}: true={int(y_np[j])}, top5={top5[j].tolist()}")
        break

preview_samples(val_ds, n_batches=1)

# Save the chosen split metadata for reproducibility (I have already created in Step 3)
print("Training complete. Looks for model files in models direcoty")
print(" - models/kb_softmax_tf.keras (best checkpoint)")
print(" - models/kb_softmax_tf_final.keras (final save)")
print(" - artifacts/kb_train_log.csv (metrics per epoch)")


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ group_ix            │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ software_ix         │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ctx (InputLayer)    │ (None, 823)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tac (InputLayer)    │ (None, 14)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ plat (InputLayer)   │ (None, 12)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_group           │ (None, 32)        │      5,824 │ group_ix[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_soft            │ (None, 32)        │     24,288 │ software_ix[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ proj_ctx (Dense)    │ (None, 128)       │    105,344 │ ctx[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ proj_tac (Dense)    │ (None, 16)        │        224 │ tac[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ proj_pl (Dense)     │ (None, 8)         │         96 │ plat[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 216)       │          0 │ emb_group[0][0],  │
│ (Concatenate)       │                   │            │ emb_soft[0][0],   │
│                     │                   │            │ proj_ctx[0][0],   │
│                     │                   │            │ proj_tac[0][0],   │
│                     │                   │            │ proj_pl[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 256)       │     55,552 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     32,896 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ logits (Dense)      │ (None, 823)       │    106,167 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 330,391 (1.26 MB)

 Trainable params: 330,391 (1.26 MB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 6.1560 - top1: 0.0138 - top10: 0.1442 - top5: 0.0780 - val_loss: 5.5596 - val_top1: 0.0332 - val_top10: 0.2155 - val_top5: 0.1309
Epoch 2/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 5.2719 - top1: 0.0340 - top10: 0.2299 - top5: 0.1360 - val_loss: 5.2778 - val_top1: 0.0332 - val_top10: 0.2243 - val_top5: 0.1298
Epoch 3/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 5.1357 - top1: 0.0480 - top10: 0.2440 - top5: 0.1534 - val_loss: 5.2083 - val_top1: 0.0490 - val_top10: 0.2295 - val_top5: 0.1512
Epoch 4/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 5.0361 - top1: 0.0586 - top10: 0.2598 - top5: 0.1636 - val_loss: 5.1464 - val_top1: 0.0518 - val_top10: 0.2442 - val_top5: 0.1550
Epoch 5/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.9355 - top1: 0.0710 - top10: 0.2776 - top5: 0.1858 - val_loss: 5.0889 - val_top1: 0.0584 - val_top10: 0.2624 - val_top5: 0.1774
Epoch 6/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - lo

### 🧭 Step 6 — Use the Trained Model for Org Gap Analysis

#### Why we do this (short)
Given an organisation’s **known TTPs** (from past incidents/exercises), we ask the model to predict **other likely techniques** they might be missing.  
We **filter out** already-seen TTPs, **nudge scores** for useful diversity (e.g., adds a new tactic), and return **Top-K missing** with short reasons.

#### What we do

1. **Inputs**
   - `known_ttps_ix`: techniques the org already has evidence for  
   - Optional context: recent **tactics**, **platforms**, and any **group/software** tags (if available)

2. **Get model scores**
   - Feed the context (multi-hot vectors) into the **softmax model** to get a score for **every technique**.

3. **Filter & rank**
   - **Remove** techniques in `known_ttps_ix`.  
   - Apply small **score bonuses/penalties**, for example:
     - +δ if a candidate **introduces a new tactic** not yet seen by the org
     - +δ if the candidate **matches the org’s platforms**
     - +δ if **popular** (seen widely in MITRE)
     - −δ if it’s a **near-duplicate** (e.g., sub-tech of something already present)

4. **Explain**
   - For each top suggestion, attach a **short reason**, e.g.  
     - “Co-occurs with your known T1059 set”  
     - “Adds new tactic: Privilege Escalation”  
     - “Matches platform: Windows”  
     - “Common in attacks using Group X / Software Y”

5. **Return Top-K**
   - Output a ranked list of **Top-K missing techniques** with:
     - `tech_id` / `tech_name`  
     - **score** (post-adjustments)  
     - **reason** (1–2 lines)

#### Mini checklist

-  Load **trained softmax model** and vocab maps  
-  Build **context vectors** from the org’s known TTPs (and optional tactics/platforms)  
-  Compute scores and **filter out** known techniques  
-  Apply **bonuses** (new tactic, platform match, popularity)  
-  Return **Top-K** with concise **reasons**


In [8]:
import os, json
import numpy as np
import pandas as pd
import tensorflow as tf

# ---- paths ----
DATA = "attack-stix-data/enterprise-attack/enterprise-attack.json"
ARTI = "artifacts"
MODEL = "models/kb_softmax_tf.keras"

# ---- load model ----
model = tf.keras.models.load_model(MODEL)

# ---- load vocabs ----
with open(f"{ARTI}/tech2ix.json","r") as f: tech2ix = json.load(f)
with open(f"{ARTI}/ix2tech.json","r") as f: ix2tech = json.load(f)
with open(f"{ARTI}/tac2ix.json","r")  as f: tac2ix  = json.load(f)
with open(f"{ARTI}/plat2ix.json","r") as f: plat2ix = json.load(f)
with open(f"{ARTI}/grp2ix.json","r")  as f: grp2ix  = json.load(f)
with open(f"{ARTI}/sft2ix.json","r")  as f: sft2ix  = json.load(f)

D_TEC = len(tech2ix); D_TAC = len(tac2ix); D_PLT = len(plat2ix)

# ---- STIX helpers to rebuild technique -> tactics/names (needed for reasons) ----
def ext_id(o):
    for ref in o.get("external_references", []) or []:
        if ref.get("source_name") in ("mitre-attack","mitre-ics-attack","mitre-mobile-attack"):
            if "external_id" in ref: return ref["external_id"]
    return None

with open(DATA, "r") as f:
    bundle = json.load(f)
objs = bundle.get("objects", [])
by_id = {o.get("id"): o for o in objs if isinstance(o, dict) and o.get("id")}

# tactic shortname -> external id, and reverse map for pretty names
tactic_short2id = {}
tacid2name = {}
for o in objs:
    if o.get("type") == "x-mitre-tactic":
        taid = ext_id(o)
        if taid:
            tactic_short2id[o.get("x_mitre_shortname")] = taid
            tacid2name[taid] = o.get("name","")

# technique → {tactic ids}, technique → name
tech2tactics = {}
tech2name = {}
for o in objs:
    if o.get("type") == "attack-pattern":
        tid = ext_id(o)
        if not tid: 
            continue
        tech2name[tid] = o.get("name","")
        tacs = set()
        for kp in o.get("kill_chain_phases", []) or []:
            if kp.get("kill_chain_name") in ("mitre-attack","mitre-enterprise-attack","mitre-mobile-attack"):
                short = kp.get("phase_name")
                taid  = tactic_short2id.get(short)
                if taid: tacs.add(taid)
        tech2tactics[tid] = tacs

def parent_of(tid):  # T1059.001 -> T1059
    return tid.split(".")[0] if "." in tid else None

# ---- encoding helpers ----
def multi_hot(ixs, size):
    v = np.zeros(size, dtype=np.float32)
    for i in set(ixs or []):
        if 0 <= i < size: v[i] = 1.0
    return v

# ---- main: recommend missing TTPs for an org ----
def recommend_missing_for_org(known_ttp_ids, top_k=10, new_tactic_bonus=0.10, parent_bonus=0.05):
    # encode context
    seen_ix = [tech2ix[t] for t in known_ttp_ids if t in tech2ix]
    ctx = multi_hot(seen_ix, D_TEC)

    # derive tactics covered by the org's seen set
    seen_tacs = set()
    for t in known_ttp_ids:
        seen_tacs |= tech2tactics.get(t, set())

    # build Keras inputs (group/software unknown here → 0 bucket)
    xb = {
        "group_ix":    np.array([0], dtype="int32"),
        "software_ix": np.array([0], dtype="int32"),
        "ctx":         np.expand_dims(ctx, 0),
        "tac":         np.expand_dims(multi_hot([tac2ix[ta] for ta in seen_tacs if ta in tac2ix], D_TAC), 0),
        "plat":        np.expand_dims(np.zeros(D_PLT, dtype=np.float32), 0)
    }

    logits = model(xb, training=False).numpy()[0]
    probs  = tf.nn.softmax(logits).numpy()

    # mask out already-seen techniques
    for ix in seen_ix:
        probs[ix] = -1.0

    # re-rank bonuses
    final_scores = probs.copy()
    for ix, p in enumerate(probs):
        if p < 0:  # seen
            continue
        tid = ix2tech[str(ix)]
        # new tactic bonus
        new_tacs = tech2tactics.get(tid, set()) - seen_tacs
        if new_tacs:
            final_scores[ix] += new_tactic_bonus * len(new_tacs)
        # parent bonus
        if parent_of(tid) is None:
            final_scores[ix] += parent_bonus

    # top-k
    top_ix = np.argsort(-final_scores)[:top_k]
    out = []
    for ix in top_ix:
        tid = ix2tech[str(ix)]
        tacs = [tacid2name.get(ta, ta) for ta in sorted(tech2tactics.get(tid, set()))]
        reason_bits = []
        new_tacs = tech2tactics.get(tid, set()) - seen_tacs
        if new_tacs:
            reason_bits.append(f"adds new tactic(s): {', '.join([tacid2name.get(ta, ta) for ta in new_tacs])}")
        if parent_of(tid) is None:
            reason_bits.append("parent technique")
        reason = "; ".join(reason_bits) if reason_bits else "co-occurs in MITRE with similar contexts"
        out.append({
            "tech_id": tid,
            "tech_name": tech2name.get(tid, ""),
            "tactics": tacs,
            "score": float(final_scores[ix])
        })
    return out

# ---- example usage: plug in an org's known TTP list ----
# e.g., known from orgs_full.csv row like "T1078;T1566;T1059"
example_known = ["T1105","T1587", "T1583.001"]

recs = recommend_missing_for_org(example_known, top_k=10)
for i, r in enumerate(recs, 1):
    print(f"{i}. {r['tech_id']} — {r['tech_name']} | tactics: {', '.join(r['tactics'])}")


1. T1078 — Valid Accounts | tactics: Initial Access, Persistence, Privilege Escalation, Defense Evasion
2. T1078.003 — Local Accounts | tactics: Initial Access, Persistence, Privilege Escalation, Defense Evasion
3. T1078.002 — Domain Accounts | tactics: Initial Access, Persistence, Privilege Escalation, Defense Evasion
4. T1078.001 — Default Accounts | tactics: Initial Access, Persistence, Privilege Escalation, Defense Evasion
5. T1078.004 — Cloud Accounts | tactics: Initial Access, Persistence, Privilege Escalation, Defense Evasion
6. T1574 — Hijack Execution Flow | tactics: Persistence, Privilege Escalation, Defense Evasion
7. T1556 — Modify Authentication Process | tactics: Persistence, Defense Evasion, Credential Access
8. T1053 — Scheduled Task/Job | tactics: Execution, Persistence, Privilege Escalation
9. T1150 — Plist Modification | tactics: Persistence, Privilege Escalation, Defense Evasion
10. T1152 — Launchctl | tactics: Execution, Persistence, Defense Evasion


In [11]:
# ---- Step 6 — Use mapped org_ttp_map.csv to predict missing TTPs for each organisation ----

# Before this lets map the organisation TTPs with MITRE TTPs code
from mitre_ttp_mapping import map_org_ttp

map_org_ttp(
    org_csv="orgs_full.csv",
    stix_path="attack-stix-data/enterprise-attack/enterprise-attack.json",
    ttp_col="TTPs",
    exid_col="ORGID",
    out_map_csv="org_ttp_map.csv",
    out_unmatched_csv="org_ttps_unmatched.csv"
)

# ---- load org-TTP mapping file ----
df_map = pd.read_csv("org_ttp_map.csv")
org_ids = sorted(df_map["ORGID"].unique())

def get_ttps_for_org(orgid):
    ttps = df_map.loc[df_map["ORGID"]==orgid, "attack_id"].dropna().unique().tolist()
    return [t for t in ttps if t.startswith("T")]  # ignore TA tactic IDs for now

def run_for_org(orgid, top_k=10):
    known_ttps = get_ttps_for_org(orgid)
    if not known_ttps:
        print(f"\nORG {orgid}: no techniques found")
        return
    recs = recommend_missing_for_org(known_ttps, top_k=top_k)
    print(f"\n=== ORG {orgid} ===")
    print("Known TTPs:", ", ".join(known_ttps))
    print("Predicted missing TTPs:")
    for i, r in enumerate(recs, 1):
        print(f"{i}. {r['tech_id']} — {r['tech_name']} | tactics: {', '.join(r['tactics'])}")

# ---- example: run for one org ----
# run_for_org(orgid=12, top_k=10)

# ---- or loop over all ----
for oid in org_ids:
    run_for_org(oid, top_k=10)


[OK] wrote org_ttp_map.csv and org_ttps_unmatched.csv

=== ORG 1 ===
Known TTPs: T1218, T1046
Predicted missing TTPs:
1. T1078 — Valid Accounts | tactics: Initial Access, Persistence, Privilege Escalation, Defense Evasion
2. T1053 — Scheduled Task/Job | tactics: Execution, Persistence, Privilege Escalation
3. T1179 — Hooking | tactics: Persistence, Privilege Escalation, Credential Access
4. T1053.005 — Scheduled Task | tactics: Execution, Persistence, Privilege Escalation
5. T1078.003 — Local Accounts | tactics: Initial Access, Persistence, Privilege Escalation, Defense Evasion
6. T1078.002 — Domain Accounts | tactics: Initial Access, Persistence, Privilege Escalation, Defense Evasion
7. T1053.003 — Cron | tactics: Execution, Persistence, Privilege Escalation
8. T1053.002 — At | tactics: Execution, Persistence, Privilege Escalation
9. T1078.001 — Default Accounts | tactics: Initial Access, Persistence, Privilege Escalation, Defense Evasion
10. T1078.004 — Cloud Accounts | tactics: Init